In [1]:
import numpy as np
import pandas as pd

from scipy.optimize import curve_fit
from scipy.optimize import minimize

import matplotlib.pyplot as plt

import time

In [2]:
df_k = pd.read_csv('N2.csv')

In [5]:
def mape_loss(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    # print(y_true, y_pred)

    errors = np.abs((y_true - y_pred) / y_true)

    mape = np.mean(errors) * 100
    
    return mape

In [34]:
def k_physical_model_flex(T, *coeffs, deg_A=2, deg_B=2, deg_C=2):
    """
    Гибкая версия с разными степенями полиномов
    """
    x = T**(-1/3)
    
    # Разделяем коэффициенты
    idx = 0
    
    # A(x)
    A = 0.0
    for k in range(deg_A + 1):
        A += coeffs[idx] * x**k
        idx += 1
    
    # B(x)
    B = 0.0
    for k in range(deg_B + 1):
        B += coeffs[idx] * x**k
        idx += 1
    
    # C(x)
    # C = 0.0
    # for k in range(deg_C + 1):
    #     C += coeffs[idx] * x**k
    #     idx += 1
    
    k = A * np.exp(B) #+ C
    return np.log(k) # защита от log(0)

def fit_physical_model(df, deg_A=2, deg_B=2, deg_C=2):
    """
    Обучение физической модели
    """
    total_params = (deg_A + 1) + (deg_B + 1)# + (deg_C + 1)
    
    params_arr = []
    pred_df = pd.DataFrame(index=df.index)
    
    for i in range(1, df.shape[1]):
        print(f"\nРеакция {i}:")
        
        y_log = np.log(df.iloc[:, i].values)
        # y_log = df.iloc[:, i].values
        
        # Начальное приближение
        p0 = np.ones(total_params)
        
        # A(x) коэффициенты: предполагаем A ~ 1
        p0[:deg_A+1] = [1.0] + [0.0]*deg_A
        
        # B(x) коэффициенты: B ~ -E/RT (отрицательные значения)
        p0[deg_A+1:deg_A+deg_B+2] = [-10.0] + [0.0]*deg_B
        
        # C(x) коэффициенты: C ~ 0 (малая поправка)
        #p0[deg_A+deg_B+2:] = [0.0]*(deg_C+1)
        
        try:
            # Обучение в логарифмической шкале
            params, pcov = curve_fit(lambda T, *c: k_physical_model_flex(T, *c, deg_A=deg_A, deg_B=deg_B, deg_C=deg_C), df.iloc[:, 0], y_log, p0=p0, method='lm')
            
            # Предсказание
            log_k_pred = k_physical_model_flex(df.iloc[:, 0], *params,
                                              deg_A=deg_A, deg_B=deg_B, deg_C=deg_C)
            k_pred = np.exp(log_k_pred)
            # k_pred = log_k_pred
            
            mape_val = mape_loss(y_log, k_pred)
            
            print(f"  Параметров: {total_params}")
            print(f"  MAPE: {mape_val:.6f}")
            
            # Анализ параметров
            print("  Коэффициенты A(x):", params[:deg_A+1])
            print("  Коэффициенты B(x):", params[deg_A+1:deg_A+deg_B+2])
            print("  Коэффициенты C(x):", params[deg_A+deg_B+2:])
            
            params_arr.append(params)
            pred_df[i-1] = k_pred
            
        except Exception as e:
            print(f"  Ошибка: {e}")
            # Fallback: простая модель
            params_simple = [-10.0, -1000.0]  # a + b/T
            params_arr.append(params_simple)
            pred_df[i-1] = np.exp(params_simple[0] + params_simple[1]/df.iloc[:, 0])
    
    return params_arr, pred_df

In [35]:
params, pred = fit_physical_model(df_k)


Реакция 1:


C:\anaconda\envs\neural\lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: overflow encountered in exp
  result = getattr(ufunc, method)(*inputs, **kwargs)
C:\anaconda\envs\neural\lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


  Параметров: 6
  MAPE: 100.000000
  Коэффициенты A(x): [ 0.34568914 -6.54821836 35.0391747 ]
  Коэффициенты B(x): [ -20.42426231  -34.64315404 -739.68558934]
  Коэффициенты C(x): []

Реакция 2:
  Параметров: 6
  MAPE: 100.000000
  Коэффициенты A(x): [ 0.31897496 -6.0373357  32.26422476]
  Коэффициенты B(x): [ -19.77402337  -30.76164574 -742.89739018]
  Коэффициенты C(x): []

Реакция 3:
  Параметров: 6
  MAPE: 100.000000
  Коэффициенты A(x): [ 0.31549232 -5.96560971 31.83879755]
  Коэффициенты B(x): [ -19.48084489  -26.93342171 -745.63223372]
  Коэффициенты C(x): []

Реакция 4:
  Параметров: 6
  MAPE: 100.000000
  Коэффициенты A(x): [ 0.31718689 -5.99075065 31.92929114]
  Коэффициенты B(x): [ -19.321533    -23.16538775 -747.84862089]
  Коэффициенты C(x): []

Реакция 5:


C:\anaconda\envs\neural\lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: overflow encountered in exp
  result = getattr(ufunc, method)(*inputs, **kwargs)
C:\anaconda\envs\neural\lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
C:\anaconda\envs\neural\lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: overflow encountered in exp
  result = getattr(ufunc, method)(*inputs, **kwargs)
C:\anaconda\envs\neural\lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
C:\anaconda\envs\neural\lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: overflow encountered in exp
  result = getattr(ufunc, method)(*inputs, **kwargs)
C:\anaconda\envs\neural\lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc

  Параметров: 6
  MAPE: 100.000000
  Коэффициенты A(x): [ 0.3315719  -6.25406425 33.28559709]
  Коэффициенты B(x): [ -19.26529844  -19.46522465 -749.50086352]
  Коэффициенты C(x): []

Реакция 6:
  Параметров: 6
  MAPE: 100.000000
  Коэффициенты A(x): [ 0.31175913 -5.87133659 31.20299899]
  Коэффициенты B(x): [ -19.14324568  -15.84086744 -750.54231582]
  Коэффициенты C(x): []

Реакция 7:
  Параметров: 6
  MAPE: 100.000000
  Коэффициенты A(x): [ 0.32050295 -6.02550448 31.97403917]
  Коэффициенты B(x): [ -19.13773443  -12.3006649  -750.92491046]
  Коэффициенты C(x): []

Реакция 8:
  Параметров: 6
  MAPE: 100.000000
  Коэффициенты A(x): [ 0.31975715 -5.99969729 31.78746924]
  Коэффициенты B(x): [ -19.12167872   -8.85436341 -750.59133531]
  Коэффициенты C(x): []

Реакция 9:
  Параметров: 6
  MAPE: 100.000000
  Коэффициенты A(x): [ 0.33909296 -6.34855753 33.58170073]
  Коэффициенты B(x): [ -19.18097556   -5.51121542 -749.49030129]
  Коэффициенты C(x): []


C:\anaconda\envs\neural\lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: overflow encountered in exp
  result = getattr(ufunc, method)(*inputs, **kwargs)
C:\anaconda\envs\neural\lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
C:\anaconda\envs\neural\lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
C:\anaconda\envs\neural\lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: overflow encountered in exp
  result = getattr(ufunc, method)(*inputs, **kwargs)
C:\anaconda\envs\neural\lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
C:\anaconda\envs\neural\lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: overflow encountered in exp
  result = getattr(ufunc

In [36]:
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit

def log_k_model(T, *coeffs, deg_A=1, deg_B=1):
    """
    ln k = A_tilde(x) + B(x)
    x = T^{-1/3}
    """
    x = T ** (-1/3)
    idx = 0

    # A_tilde(x)
    A_tilde = 0.0
    for k in range(deg_A + 1):
        A_tilde += coeffs[idx] * x**k
        idx += 1

    # B(x)
    B = 0.0
    for k in range(deg_B + 1):
        B += coeffs[idx] * x**k
        idx += 1

    return A_tilde + B


def fit_physical_model(df, deg_A=1, deg_B=1):
    T = df.iloc[:, 0].values
    total_params = (deg_A + 1) + (deg_B + 1)

    params_arr = []
    pred_df = pd.DataFrame(index=df.index)

    for i in range(1, df.shape[1]):
        print(f"\nРеакция {i}:")

        y = df.iloc[:, i].values
        y_log = np.log(y)

        # начальное приближение
        p0 = np.zeros(total_params)

        # физика: B < 0
        p0[deg_A + 1] = -10.0

        params, _ = curve_fit(
            lambda T, *c: log_k_model(T, *c, deg_A=deg_A, deg_B=deg_B),
            T,
            y_log,
            p0=p0,
            method='lm'
        )

        # восстановление k
        log_k_pred = log_k_model(
            T, *params, deg_A=deg_A, deg_B=deg_B
        )
        k_pred = np.exp(log_k_pred)

        mape_val = np.mean(np.abs((y - k_pred) / y)) * 100

        print(f"  Параметров: {total_params}")
        print(f"  MAPE: {mape_val:.4f}%")
        print("  A_tilde(x):", params[:deg_A+1])
        print("  B(x):", params[deg_A+1:])

        params_arr.append(params)
        pred_df[i-1] = k_pred

    return params_arr, pred_df


In [37]:
fit_physical_model(df_k)


Реакция 1:
  Параметров: 4
  MAPE: 13.9962%
  A_tilde(x): [ 702.57904964 -355.25839552]
  B(x): [-720.48321517  189.54349239]

Реакция 2:
  Параметров: 4
  MAPE: 14.2210%
  A_tilde(x): [703.45235131 -19.42340382]
  B(x): [-720.76091126 -143.04009459]

Реакция 3:
  Параметров: 4
  MAPE: 14.4401%
  A_tilde(x): [703.6008767  -92.08781419]
  B(x): [-720.60395885  -67.09629889]

Реакция 4:
  Параметров: 4
  MAPE: 14.6522%
  A_tilde(x): [703.6851283  142.61581214]
  B(x): [-720.50303362 -298.49215359]

Реакция 5:
  Параметров: 4
  MAPE: 14.8557%
  A_tilde(x): [ 2026.3361772  -5988.70012882]
  B(x): [-2043.03603476  5836.16030144]

Реакция 6:
  Параметров: 4
  MAPE: 15.0489%
  A_tilde(x): [ 703.57056842 -276.11642782]
  B(x): [-720.1958651   126.94217291]

Реакция 7:
  Параметров: 4
  MAPE: 15.2297%
  A_tilde(x): [ 703.80075333 -125.9504689 ]
  B(x): [-720.3823968   -19.82888847]

Реакция 8:
  Параметров: 4
  MAPE: 15.3962%
  A_tilde(x): [703.82591019 -75.80485429]
  B(x): [-720.38732714  -6

([array([ 702.57904964, -355.25839552, -720.48321517,  189.54349239]),
  array([ 703.45235131,  -19.42340382, -720.76091126, -143.04009459]),
  array([ 703.6008767 ,  -92.08781419, -720.60395885,  -67.09629889]),
  array([ 703.6851283 ,  142.61581214, -720.50303362, -298.49215359]),
  array([ 2026.3361772 , -5988.70012882, -2043.03603476,  5836.16030144]),
  array([ 703.57056842, -276.11642782, -720.1958651 ,  126.94217291]),
  array([ 703.80075333, -125.9504689 , -720.3823968 ,  -19.82888847]),
  array([ 703.82591019,  -75.80485429, -720.38732714,  -66.55010882]),
  array([ 703.82670673,  -96.51777454, -720.38652312,  -42.38318982])],
                0             1             2             3             4  \
 0   2.672071e-17  7.213179e-17  1.461807e-16  2.635811e-16  4.459751e-16   
 1   4.976346e-17  1.327060e-16  2.656498e-16  4.730893e-16  7.905003e-16   
 2   9.267726e-17  2.441487e-16  4.827575e-16  8.491258e-16  1.401178e-15   
 3   1.725980e-16  4.491778e-16  8.773008e-16  1